# SQD + SBD walkthrough — H2O (serial)

Same self-consistent SQD workflow as `run_sqd_sbd.py`, but interactive and serial.
Runs on a single MPI rank (mpi4py auto-initializes `MPI.COMM_WORLD` with size=1 inside a Jupyter kernel; SBD's MPI collectives become no-ops).

Workload: H2O FCIDUMP (NORB=24, NELEC=10, MS2=0), 300 uniform-random bitstrings refined toward HF, 1 batch, 2 SQD iterations.
Should converge to ~-76.2 Ha in a few seconds on CPU (FCI reference ≈ -76.24 Ha). Scale samples_per_batch up on a GPU box for a tighter result.

**Multi-rank version:** [`run_sqd_sbd.py`](./run_sqd_sbd.py) — same recipe, run with `mpirun -np N python run_sqd_sbd.py …`.

## 1. Imports + environment check

In [1]:
from functools import partial
import json, numpy as np
from pathlib import Path
from mpi4py import MPI
from pyscf import ao2mo, tools
from qiskit_addon_sqd.fermion import diagonalize_fermionic_hamiltonian
from qiskit_addon_sqd.counts import generate_bit_array_uniform

import sbd
from sbd.sbd_solver import solve_sci_batch
from sbd.device_config import DeviceConfig, print_device_info

comm = MPI.COMM_WORLD
print(f'SBD version            : {sbd.__version__}')
print(f'Compiled SBD backends  : {sbd.available_backends()}')
print(f'MPI library            : {MPI.Get_library_version().splitlines()[0]}')
print(f'MPI world size         : {comm.Get_size()}    (Jupyter kernel = single process)')
print(f'My rank                : {comm.Get_rank()}')
print()
print_device_info()

SBD version            : 1.5.0
Compiled SBD backends  : ['cpu']
MPI library            : Open MPI v5.0.9, package: Open MPI brew@Tahoe-arm64.local Distribution, ident: 5.0.9, repo rev: v5.0.9, Oct 30, 2025 
MPI world size         : 1    (Jupyter kernel = single process)
My rank                : 0

SBD Device Information
✗ No GPU detected
✓ CPU Available: Always


## 2. Load the Hamiltonian (H2O FCIDUMP)

FCIDUMP carries the one- and two-electron integrals plus orbital count and electron count. PySCF reads it; we extract `hcore` and `eri` for the SQD outer loop.

In [2]:
fcidump_path = '../../data/h2o/fcidump.txt'
norb, nelec_total, ms2 = 24, 10, 0          # h2o constants
num_elec_a = (nelec_total + ms2) // 2        # 5 alpha
num_elec_b = (nelec_total - ms2) // 2        # 5 beta

mf    = tools.fcidump.to_scf(fcidump_path)
hcore = mf.get_hcore()
eri   = ao2mo.restore(1, mf._eri, norb)
nuc   = mf.mol.energy_nuc()
print(f'NORB={norb}, nelec=({num_elec_a},{num_elec_b}), nuc={nuc:.6f}')

Parsing ../../data/h2o/fcidump.txt
NORB=24, nelec=(5,5), nuc=9.193913


## 3. Bitstrings — uniform random, refined by configuration recovery

We generate uniform-random 48-bit strings as a stand-in for noisy quantum-measurement counts, then pass HF initial occupancies to `diagonalize_fermionic_hamiltonian` (next cell). qiskit-addon-sqd's configuration-recovery step uses those occupancies to pull random samples toward physically meaningful configurations.

*(In production you'd pass actual hardware counts via `qiskit_addon_sqd.counts.bit_array_to_arrays(...)` — see `count_dict_h2o.json` for an example, and `run_sqd_sbd.py --counts ...` for the loader.)*

In [3]:
rng = np.random.default_rng(42)
bit_array = generate_bit_array_uniform(300, norb*2, rand_seed=rng)
print(f'Generated {bit_array.num_shots} uniform-random bitstrings ({norb*2} bits / shot)')

# HF initial occupancies: lowest 5 orbitals filled in each spin sector
hf_occ_a = np.array([1.0]*num_elec_a + [0.0]*(norb - num_elec_a))
hf_occ_b = np.array([1.0]*num_elec_b + [0.0]*(norb - num_elec_b))
print(f'HF α occupancy : {hf_occ_a}')
print(f'HF β occupancy : {hf_occ_b}')

Generated 300 uniform-random bitstrings (48 bits / shot)
HF α occupancy : [1. 1. 1. 1. 1. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
HF β occupancy : [1. 1. 1. 1. 1. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]


## 4. Wire SBD into qiskit-addon-sqd's `sci_solver=` slot

No `sbd.init()` and no `mpi_comm=` needed: `solve_sci_batch` auto-initializes the SBD backend on first call and falls back to `MPI.COMM_WORLD` when `mpi_comm` is omitted (in a Jupyter kernel that's the size-1 communicator).

In [4]:
sbd_config = {
    'method': 0,           # 0 = Davidson
    'eps': 1e-8,           # Davidson convergence tolerance
    'max_it': 100,         # max Davidson iterations per diagonalization
    'max_nb': 50,          # block size
    'do_rdm': 0,           # 0 = density only (sufficient for SQD)
    'do_shuffle': 0,
    'carryover_type': 1,   # singles-only carryover (stable default)
    'ratio': 0.1,
    'threshold': 1e-4,
    'bit_length': 64,
    # serial: 1×1×1 MPI sub-communicator grid
    'adet_comm_size': 1, 'bdet_comm_size': 1, 'task_comm_size': 1,
}

sbd_solver = partial(
    solve_sci_batch,
    sbd_config=sbd_config,
    device_config=DeviceConfig.cpu(),       # macOS / no-GPU notebook
    fcidump_path=fcidump_path,
)

## 5. Run the SQD self-consistent loop

Sample → configuration recovery → subsample → SBD diagonalize → carryover → repeat. 1 batch per iteration, 2 SQD iterations.

In [5]:
history = []
def callback(results):
    history.append(results)
    iteration = len(history)
    for i, r in enumerate(results):
        total_e = r.energy + nuc
        dim = np.prod(r.sci_state.amplitudes.shape)
        print(f'  iter {iteration}, batch {i}:  E = {total_e:.10f} Ha   subspace dim = {dim:_}')

result = diagonalize_fermionic_hamiltonian(
    hcore, eri, bit_array,
    norb=norb,
    nelec=(num_elec_a, num_elec_b),
    samples_per_batch=300,
    num_batches=1,
    max_iterations=2,
    initial_occupancies=(hf_occ_a, hf_occ_b),    # bootstraps recovery from HF
    sci_solver=sbd_solver,
    symmetrize_spin=True,
    callback=callback,
    seed=rng,
)

print()
print(f'Final SQD energy:  {result.energy + nuc:.10f} Ha   (electronic {result.energy:.10f} + nuc {nuc:.6f})')
print(f'FCI reference   :  ≈ −76.24 Ha')

 Elapsed time for helper construction 0.009994 (sec) 
 Elapsed time for init 0.000227 (sec) 
 Elapsed time for makeQChamDiagTerms 0.057454 (sec) 


 Davidson iteration 0.0 (tol=0.663572): -76.0268


 Davidson iteration 0.1 (tol=0.0686387): -76.1302 -72.674


 Davidson iteration 0.2 (tol=0.0145937): -76.1315 -74.0179 -72.6601


 Davidson iteration 0.3 (tol=0.00284917): -76.1316 -74.123 -73.887 -72.3602


...

 Davidson iteration 0.9 (tol=6.470811552318204e-07): -76.1914026681267 -75.38872228454233 -74.65747070191892 -74.25609166817917


 Davidson iteration 0.10 (tol=1.731023815587777e-07): -76.1914026681269 -75.53876039435609 -74.77726756654594 -74.37048324042696


 Davidson iteration 0.11 (tol=3.686977369060456e-08): -76.19140266812693 -75.55403478486762 -74.77891537447849 -74.62147750805481


 Davidson iteration 0.12 (tol=6.692097159381116e-09): -76.19140266812695 -75.59553506148016 -74.79126390457206 -74.71197476140031
 Elapsed time for davidson 4.696738 (sec) 
 Elapsed time for diagonalization 4.696743 (sec) 


 Elapsed time for mult 0.345435 (sec) 
 Energy = -76.19140266812735
 Elapsed time for measurement 0.001493 (sec) 
 truncated weight in carry-over for alpha-det = 0.0009502579461793115
 truncated weight in carry-over for beta-det = 0.0009502579461795335
  iter 2, batch 0:  E = -76.1914026681 Ha   subspace dim = 3_025

Final SQD energy:  -76.1914026681 Ha   (electronic -85.3853158288 + nuc 9.193913)
FCI reference   :  ≈ −76.24 Ha


## Notes

- **Why this works in a notebook**: SBD is MPI-native at the C++ level, but a Jupyter kernel is a single Python process. `mpi4py` auto-initializes MPI with `MPI.COMM_WORLD` of size 1; SBD's collectives all become no-ops; the full SQD loop runs on rank 0.
- **Why `initial_occupancies=` is needed for random samples**: 48-bit uniform-random strings overwhelmingly fail the Hamming-weight check (5α+5β). qiskit-addon-sqd's configuration_recovery uses occupancies to pull noisy samples toward physical configurations; HF occupancies are the natural seed before the first SBD diagonalization gives real ones.
- **Multi-rank**: launch [`run_sqd_sbd.py`](./run_sqd_sbd.py) with `mpirun -np N python …` for production scale.
- **GPU**: pass `DeviceConfig.gpu()` (Thrust) or `DeviceConfig.gpu_omp()` (LLVM offload) on a CUDA-capable machine. Not available on macOS — keep `DeviceConfig.cpu()` here.